In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from typing import Dict, List, Tuple
from data_loading import *
from loss_funcs import *

print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU              : {torch.cuda.get_device_name(0)}") # I have a 3090
    torch.set_float32_matmul_precision("medium")
    print("float32 matmul precision set to 'medium'")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
weather_cols_all = ['temperature_2m',
       'apparent_temperature', 'dew_point_2m', 'relative_humidity_2m',
       'precipitation', 'rain', 'snowfall', 'cloud_cover', 'cloud_cover_low',
       'cloud_cover_mid', 'cloud_cover_high', 'surface_pressure',
       'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m',
       'shortwave_radiation', 'diffuse_radiation', 'direct_normal_irradiance']

other_cols = [ # these are not static
    'dam_price', 'buy_bm_price', 'sell_bm_price',
    'max_power', 'max_solar', 'max_ev'
]

cat_columns = [
    'eic_code', 'dso_desc', 'station_type', 'oblast',
    'Month', 'Day', 'Hour', 'day_of_week', 'season'
]

static_cols = [
    'latitude', 'longitude', 'eic_code', 'dso_desc', 'station_type', 'oblast'
]

calendar_cols = [
    'Month', 'Day', 'Hour', 'day_of_week', 'season'
]

time_cols = ['datetime', 'time_idx']

FUTURE_REALS = weather_cols_all + calendar_cols + static_cols + other_cols
print(f"y col is: {Y_COL}, group col is: {GROUP_COL}\n"
      f"features: {FUTURE_REALS}")

In [ ]:
# this data has a data column and categorical columns are kept intact and will need to be handled.
# "time_idx" is already built in train, val and test and is continuous through them
print("Loading train …")
train = load_and_prepare(TRAIN_PATH_WITH_DATETME)

print("Loading val   …")
val = load_and_prepare(VAL_PATH_WITH_DATETME)

print("Loading test  …")
test = load_and_prepare(TEST_PATH_WITH_DATETME)

print(f"train: {train.shape}")
print(f"val : {val.shape}")
print(f"test: {test.shape}")

In [ ]:
# # If a model cant handle categorical columns natively, or through embedings use this data
# # It has no datetime column and all columns are numeric, as all cat column were ohe
# # GROUP_COL is the only exception, and is not ohe. Ohe it before training
# print("Loading train …")
# train = load_and_prepare(TRAIN_PATH_OHE)
#
# print("Loading val   …")
# val = load_and_prepare(VAL_PATH_OHE)
#
# print("Loading test  …")
# test = load_and_prepare(TEST_PATH_OHE)
#
# print(f"train: {train.shape}")
# print(f"val  : {val.shape}")
# print(f"test : {test.shape}")

In [ ]:
# this cell samples locations, I'll use it if training takes too long, otherwise don't touch it

TARGET_STATIONS = 395

station_stats = (
    train.groupby(GROUP_COL)
    .agg(rows=(Y_COL, "count"))
    .reset_index()
    .sort_values("rows", ascending=False)
)

sampled_stations = station_stats.sample(
    n=TARGET_STATIONS, random_state=42
)[GROUP_COL].values

print(f"Stations: {len(sampled_stations)}")

train = train[train[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)
val   = val[val[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)
test  = test[test[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)

print(f"Train rows : {len(train):,}")
print(f"Val rows   : {len(val):,}")
print(f"Test rows  : {len(test):,}")

In [ ]:
training_cutoff = train["time_idx"].max()
val_cutoff      = val["time_idx"].max()
test_cutoff     = test["time_idx"].max()

print(f"training cutoff : {training_cutoff}")
print(f"val cutoff      : {val_cutoff}")
print(f"test cutoff     : {test_cutoff}")

In [ ]:
# =============================================================================
# N-HiTS configuration
# =============================================================================
# We implement N-HiTS as a custom PyTorch model + training loop. NeuralForecast
# does not cleanly accept a custom loss that needs extra per-timestep tensors
# (the price columns), so a custom loop is the cleanest path that preserves the
# money_pct + 0.2 * smape objective without hacks.
#
# N-HiTS = stacks of MLP blocks. Each stack:
#   1) Multi-rate pooling on the lookback window (downsamples by k_pool).
#      Different stacks use different pool sizes -> hierarchical decomposition.
#   2) An MLP produces backcast (theta_b) + forecast (theta_f) coefficients.
#   3) Hierarchical interpolation upsamples theta_f to the full horizon length
#      (different stacks output forecasts at different temporal resolutions,
#      from coarse/long-range to fine/short-range).
# The residual after each block's backcast is fed to the next block, and
# forecasts are summed across blocks/stacks.
# =============================================================================

INPUT_SIZE   = 168     # lookback window: one week of hourly data
HORIZON      = 48      # forecast horizon: 48 hours
BATCH_SIZE   = 256
MAX_EPOCHS   = 20
LR           = 1e-3
WEIGHT_DECAY = 1e-5
SMAPE_WEIGHT = 0.2     # combined_loss = money_pct + 0.2 * smape
GRAD_CLIP    = 1.0
NUM_WORKERS  = 4

# N-HiTS structural hyperparameters (hierarchical interp + multi-rate pooling).
# 3 stacks: low / medium / high frequency. Pool sizes & interpolation rates form
# the hierarchy.
N_STACKS         = 3
N_BLOCKS         = 1                  # blocks per stack
HIDDEN_SIZE      = 512
N_LAYERS         = 2                  # MLP depth inside each block
POOL_KERNELS     = [8, 4, 1]          # one per stack: coarse -> fine
N_FREQ_DOWNSAMPLE= [24, 8, 1]         # forecast theta length = HORIZON / this
                                      # then interpolated up to HORIZON
DROPOUT          = 0.0

print(f"INPUT_SIZE={INPUT_SIZE}, HORIZON={HORIZON}, "
      f"stacks={N_STACKS}, hidden={HIDDEN_SIZE}, "
      f"pool_kernels={POOL_KERNELS}, n_freq_downsample={N_FREQ_DOWNSAMPLE}")
print(f"Combined loss = money_pct + {SMAPE_WEIGHT} * smape")

In [ ]:
# =============================================================================
# Feature preparation for N-HiTS
# =============================================================================
# Strategy:
#   - target series:   sum_of_kWh, lookback window
#   - future exogenous: weather + calendar + prices (known for the horizon)
#   - static features:  per-station (latitude, longitude, station_type, ...)
# We encode categoricals via integer codes (cheap embeddings later if needed),
# then standardize numerics. Prices are kept un-standardized in a SEPARATE
# tensor because they are passed straight into the money loss as-is.
# =============================================================================

PRICE_COLS    = ["dam_price", "sell_bm_price", "buy_bm_price"]
TIME_VARYING_NUMERIC = [c for c in (weather_cols_all + other_cols) if c not in PRICE_COLS]
CALENDAR_NUMERIC     = calendar_cols                       # already integers
STATIC_NUMERIC       = ["latitude", "longitude"]
STATIC_CATEGORICAL   = ["dso_desc", "station_type", "oblast"]  # eic_code is the group id, used separately

# Build categorical -> int code maps from train (stable across val/test)
def _build_codes(df: pd.DataFrame, cols: List[str]) -> Dict[str, Dict]:
    return {c: {v: i for i, v in enumerate(df[c].astype(str).unique())} for c in cols}

cat_maps = _build_codes(train, STATIC_CATEGORICAL)

def _apply_codes(df: pd.DataFrame, maps: Dict[str, Dict]) -> pd.DataFrame:
    df = df.copy()
    for c, m in maps.items():
        df[c] = df[c].astype(str).map(m).fillna(-1).astype(np.int64)
    return df

train = _apply_codes(train, cat_maps)
val   = _apply_codes(val,   cat_maps)
test  = _apply_codes(test,  cat_maps)

# Standardize time-varying numeric features and the target using train stats.
# Prices are NOT standardized - they go into the money loss in original scale.
feat_means = train[TIME_VARYING_NUMERIC].astype(np.float32).mean()
feat_stds  = train[TIME_VARYING_NUMERIC].astype(np.float32).std().replace(0, 1.0)
y_mean     = float(train[Y_COL].mean())
y_std      = float(train[Y_COL].std()) or 1.0
static_means = train[STATIC_NUMERIC].astype(np.float32).mean()
static_stds  = train[STATIC_NUMERIC].astype(np.float32).std().replace(0, 1.0)

def _standardize(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df[TIME_VARYING_NUMERIC] = (df[TIME_VARYING_NUMERIC].astype(np.float32) - feat_means) / feat_stds
    df[STATIC_NUMERIC]       = (df[STATIC_NUMERIC].astype(np.float32) - static_means) / static_stds
    df[Y_COL]                = (df[Y_COL].astype(np.float32) - y_mean) / y_std
    return df

train_s = _standardize(train)
val_s   = _standardize(val)
test_s  = _standardize(test)

# Number of exogenous channels used as future covariates inside N-HiTS.
# (target lookback contributes one extra channel, see model below.)
EXOG_NUMERIC = TIME_VARYING_NUMERIC + CALENDAR_NUMERIC + STATIC_NUMERIC + STATIC_CATEGORICAL
EXOG_PRICES  = PRICE_COLS  # kept aside for the loss

print(f"Numeric exogenous channels: {len(TIME_VARYING_NUMERIC)}")
print(f"Calendar channels        : {len(CALENDAR_NUMERIC)}")
print(f"Static channels          : {len(STATIC_NUMERIC) + len(STATIC_CATEGORICAL)}")
print(f"Price channels (loss only): {len(PRICE_COLS)}")
print(f"y_mean={y_mean:.3f}, y_std={y_std:.3f}")

In [ ]:
# =============================================================================
# Windowed dataset for N-HiTS
# =============================================================================
# For each station we have a continuous hourly series with a continuous
# `time_idx`. We build sliding windows of (INPUT_SIZE lookback + HORIZON future).
# Each sample yields:
#   x_lookback : [INPUT_SIZE]                    standardized target history
#   x_future   : [HORIZON, n_exog]               standardized future covariates
#   y_true     : [HORIZON]                       standardized target future
#   y_true_raw : [HORIZON]                       ORIGINAL kWh future (for money loss)
#   prices     : [HORIZON, 3]                    dam, sell_bm, buy_bm  (raw)
#
# Why both y_true and y_true_raw? Training MSE/SMAPE wants standardized values
# for stable optimization scale; the money loss must use real kWh and real
# prices. The model output is in standardized space; we de-standardize before
# the money loss.
# =============================================================================

class NHITSWindowDataset(Dataset):
    def __init__(self, df_std: pd.DataFrame, df_raw: pd.DataFrame,
                 input_size: int, horizon: int):
        assert (df_std[GROUP_COL].values == df_raw[GROUP_COL].values).all()
        self.input_size = input_size
        self.horizon    = horizon

        # Build per-station contiguous arrays sorted by time_idx.
        self.windows: List[Tuple[np.ndarray, ...]] = []  # holds (y_std, y_raw, exog, prices) per station
        self.index: List[Tuple[int, int]] = []           # (station_idx, start_pos)

        # Sort once
        df_std = df_std.sort_values([GROUP_COL, "time_idx"]).reset_index(drop=True)
        df_raw = df_raw.sort_values([GROUP_COL, "time_idx"]).reset_index(drop=True)

        exog_arr_full = df_std[EXOG_NUMERIC].to_numpy(dtype=np.float32)
        y_std_full    = df_std[Y_COL].to_numpy(dtype=np.float32)
        y_raw_full    = df_raw[Y_COL].to_numpy(dtype=np.float32)
        prices_full   = df_raw[PRICE_COLS].to_numpy(dtype=np.float32)
        groups_full   = df_std[GROUP_COL].to_numpy()

        # Find contiguous group spans
        change = np.r_[0, np.where(groups_full[1:] != groups_full[:-1])[0] + 1, len(groups_full)]
        for s_idx in range(len(change) - 1):
            a, b = change[s_idx], change[s_idx + 1]
            n = b - a
            if n < input_size + horizon:
                continue
            self.windows.append((
                y_std_full[a:b],
                y_raw_full[a:b],
                exog_arr_full[a:b],
                prices_full[a:b],
            ))
            station_id = len(self.windows) - 1
            # one window per starting position
            for start in range(0, n - input_size - horizon + 1):
                self.index.append((station_id, start))

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        sid, start = self.index[i]
        y_std, y_raw, exog, prices = self.windows[sid]
        l, h = self.input_size, self.horizon
        x_lookback = y_std[start : start + l]                    # [L]
        y_true     = y_std[start + l : start + l + h]            # [H]
        y_true_raw = y_raw[start + l : start + l + h]            # [H]
        x_future   = exog[start + l : start + l + h]             # [H, n_exog]
        prc        = prices[start + l : start + l + h]           # [H, 3]
        return (
            torch.from_numpy(x_lookback),
            torch.from_numpy(x_future),
            torch.from_numpy(y_true),
            torch.from_numpy(y_true_raw),
            torch.from_numpy(prc),
        )

train_ds = NHITSWindowDataset(train_s, train, INPUT_SIZE, HORIZON)
val_ds   = NHITSWindowDataset(val_s,   val,   INPUT_SIZE, HORIZON)
# test windows are evaluated via the recursive inference path below.

print(f"train windows : {len(train_ds):,}")
print(f"val   windows : {len(val_ds):,}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

In [ ]:
# =============================================================================
# Custom differentiable losses (PyTorch only, no NumPy, no .detach())
# =============================================================================
# money_pct_loss replicates the user's simplified market rule:
#   - if |pred - true| within +/- 10% of true, we settle at DAM price (no penalty)
#   - over-consumption beyond 10%: extra cost ~ +40% on the excess
#   - under-consumption beyond 10%: sell-back at -40% on the deficit
# The exact shape mirrors loss_funcs.money_pct() but runs on tensors with grad.
# We compute it in ORIGINAL kWh units, so we de-standardize y_pred/y_true_raw
# is already raw. y_pred (model output) is standardized -> rescale here.
# =============================================================================

EPS = 1e-8

def smape_loss_torch(y_true: torch.Tensor, y_pred: torch.Tensor) -> torch.Tensor:
    """Differentiable SMAPE in [0, 200], mean over all elements."""
    num = torch.abs(y_pred - y_true)
    den = (torch.abs(y_true) + torch.abs(y_pred)) / 2.0 + EPS
    return torch.mean(num / den) * 100.0


def money_pct_loss_torch(
    y_true_raw: torch.Tensor,   # [B, H] in real kWh
    y_pred_raw: torch.Tensor,   # [B, H] in real kWh
    dam_price:  torch.Tensor,   # [B, H]
    sell_price: torch.Tensor,   # [B, H]
    buy_price:  torch.Tensor,   # [B, H]
    band: float = 0.10,
    over_penalty:  float = 0.40,
    under_penalty: float = 0.40,
) -> torch.Tensor:
    """
    Money percent loss: differentiable PyTorch version of money_pct().
    Returns a percentage (0-100+) so it is on the same scale as SMAPE-ish.
    """
    # Reference baseline cost = paying DAM price for true consumption.
    baseline = torch.abs(y_true_raw) * dam_price + EPS  # avoid /0

    diff = y_pred_raw - y_true_raw
    band_kwh = band * torch.abs(y_true_raw)

    # over-consumption: pred > true beyond band -> buy excess at buy_price * (1+penalty)
    over_amt  = torch.clamp(diff - band_kwh, min=0.0)
    over_cost = over_amt * buy_price * (1.0 + over_penalty)

    # under-consumption: pred < true beyond band -> sell deficit at sell_price * (1-penalty)
    # (penalty here is a discount on what we earn back, so it is a *cost* relative to baseline)
    under_amt  = torch.clamp(-diff - band_kwh, min=0.0)
    under_cost = under_amt * sell_price * under_penalty  # the discounted sell-back is a cost vs baseline

    extra_cost = over_cost + under_cost

    # Per-sample percent of baseline. Mean over batch.
    pct = extra_cost.sum(dim=1) / baseline.sum(dim=1) * 100.0
    return pct.mean()


def combined_loss(
    y_pred_std: torch.Tensor,
    y_true_std: torch.Tensor,
    y_true_raw: torch.Tensor,
    prices: torch.Tensor,        # [B, H, 3]: dam, sell, buy
    y_mean_t: torch.Tensor,
    y_std_t:  torch.Tensor,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Returns (combined, money_pct, smape) — combined keeps grad."""
    # de-standardize predictions to real kWh for the money loss
    y_pred_raw = y_pred_std * y_std_t + y_mean_t

    dam  = prices[..., 0]
    sell = prices[..., 1]
    buy  = prices[..., 2]

    m_pct = money_pct_loss_torch(y_true_raw, y_pred_raw, dam, sell, buy)
    sm    = smape_loss_torch(y_true_raw, y_pred_raw)
    total = m_pct + SMAPE_WEIGHT * sm
    return total, m_pct, sm

In [ ]:
# =============================================================================
# N-HiTS model (PyTorch, paper-faithful core)
# =============================================================================
# Reference: Challu et al., "N-HiTS: Neural Hierarchical Interpolation for Time
# Series Forecasting" (2022).
#
# Key ideas implemented here:
#   * Multi-rate signal sampling: each block downsamples the lookback with
#     AvgPool1d(kernel=k_l).  Stacks with larger k_l see only low-frequency
#     content; stacks with k_l=1 see full resolution.
#   * Hierarchical interpolation: each block outputs a SHORT theta_f of length
#     ceil(H / r_l), then upsamples to H via linear interpolation.  Blocks with
#     large r_l contribute coarse/long-range structure; r_l=1 contributes fine
#     detail.  Forecasts sum across blocks/stacks.
#   * Doubly residual stacking: block i sees the input minus block i-1's
#     backcast; forecasts are accumulated.
#
# Future exogenous features are concatenated to the lookback target via a
# small projection, so the MLP gets target_history + a horizon-summary
# embedding of future covariates.  Static features are tiled into the same
# horizon-summary embedding.  This keeps the MLP shape simple while still
# letting weather/prices/calendar inform the forecast.
# =============================================================================

class NHITSBlock(nn.Module):
    def __init__(self, input_size: int, horizon: int, exog_summary_size: int,
                 hidden_size: int, n_layers: int, pool_kernel: int,
                 n_freq_downsample: int, dropout: float = 0.0):
        super().__init__()
        self.input_size  = input_size
        self.horizon     = horizon
        self.pool_kernel = max(1, pool_kernel)
        self.n_freq_down = max(1, n_freq_downsample)
        self.theta_f_size = max(1, (horizon + self.n_freq_down - 1) // self.n_freq_down)

        # AvgPool downsamples the target lookback
        self.pool = nn.AvgPool1d(kernel_size=self.pool_kernel,
                                 stride=self.pool_kernel,
                                 ceil_mode=True)
        pooled_input_size = (input_size + self.pool_kernel - 1) // self.pool_kernel

        # MLP input = pooled lookback + exog summary
        layers = []
        in_dim = pooled_input_size + exog_summary_size
        for _ in range(n_layers):
            layers += [nn.Linear(in_dim, hidden_size), nn.ReLU()]
            if dropout > 0:
                layers += [nn.Dropout(dropout)]
            in_dim = hidden_size
        self.mlp = nn.Sequential(*layers)

        # Two heads: backcast (full input length) + forecast theta (shorter, will be interpolated)
        self.backcast_head = nn.Linear(hidden_size, input_size)
        self.forecast_head = nn.Linear(hidden_size, self.theta_f_size)

    def forward(self, x_lookback: torch.Tensor, exog_summary: torch.Tensor):
        # x_lookback: [B, L] ; exog_summary: [B, E]
        pooled = self.pool(x_lookback.unsqueeze(1)).squeeze(1)            # [B, L_pool]
        h = self.mlp(torch.cat([pooled, exog_summary], dim=1))            # [B, hidden]
        backcast = self.backcast_head(h)                                  # [B, L]
        theta_f  = self.forecast_head(h)                                  # [B, theta_f_size]
        # Hierarchical interpolation: linearly upsample theta_f to horizon
        forecast = F.interpolate(theta_f.unsqueeze(1), size=self.horizon,
                                 mode="linear", align_corners=False).squeeze(1)
        return backcast, forecast


class NHITS(nn.Module):
    def __init__(self,
                 input_size: int,
                 horizon: int,
                 n_exog_per_step: int,
                 n_static: int,
                 n_stacks: int,
                 n_blocks_per_stack: int,
                 hidden_size: int,
                 n_layers: int,
                 pool_kernels: List[int],
                 n_freq_downsample: List[int],
                 dropout: float = 0.0,
                 exog_summary_dim: int = 64):
        super().__init__()
        assert len(pool_kernels) == n_stacks
        assert len(n_freq_downsample) == n_stacks

        # Encode future exogenous features into a fixed-size summary that all
        # blocks consume. Keeps block MLP input dim manageable regardless of
        # horizon and feature count.
        self.exog_proj = nn.Sequential(
            nn.Linear(horizon * n_exog_per_step + n_static, exog_summary_dim),
            nn.ReLU(),
            nn.Linear(exog_summary_dim, exog_summary_dim),
        )

        blocks = []
        for s in range(n_stacks):
            for _ in range(n_blocks_per_stack):
                blocks.append(NHITSBlock(
                    input_size=input_size,
                    horizon=horizon,
                    exog_summary_size=exog_summary_dim,
                    hidden_size=hidden_size,
                    n_layers=n_layers,
                    pool_kernel=pool_kernels[s],
                    n_freq_downsample=n_freq_downsample[s],
                    dropout=dropout,
                ))
        self.blocks = nn.ModuleList(blocks)

    def forward(self, x_lookback: torch.Tensor, x_future: torch.Tensor):
        """
        x_lookback : [B, L]
        x_future   : [B, H, n_exog]   (last n_static channels are static, repeated)
        """
        B, H, _ = x_future.shape
        flat_future = x_future.reshape(B, -1)                              # [B, H*n_exog]
        # static channels are constant across H; we still pass the flattened tensor.
        # (n_static arg of __init__ kept at 0; static cols are inside x_future.)
        exog_summary = self.exog_proj(flat_future)                         # [B, E]

        residual = x_lookback
        forecast = torch.zeros(B, H, device=x_lookback.device,
                               dtype=x_lookback.dtype)
        for blk in self.blocks:
            backcast, f_part = blk(residual, exog_summary)
            residual = residual - backcast
            forecast = forecast + f_part
        return forecast


# Build the model
N_EXOG_PER_STEP = len(EXOG_NUMERIC)
model = NHITS(
    input_size=INPUT_SIZE,
    horizon=HORIZON,
    n_exog_per_step=N_EXOG_PER_STEP,
    n_static=0,                                  # static cols already inside x_future
    n_stacks=N_STACKS,
    n_blocks_per_stack=N_BLOCKS,
    hidden_size=HIDDEN_SIZE,
    n_layers=N_LAYERS,
    pool_kernels=POOL_KERNELS,
    n_freq_downsample=N_FREQ_DOWNSAMPLE,
    dropout=DROPOUT,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"NHITS params: {n_params:,}")
print(model)

In [ ]:
# =============================================================================
# MLflow setup
# =============================================================================
import mlflow
import mlflow.pytorch

mlflow.set_experiment("electricity_nhits")
mlflow_run = mlflow.start_run(run_name="NHITS_money_pct_smape")

mlflow.log_params({
    "model":              "NHITS",
    "loss_formula":       f"money_pct + {SMAPE_WEIGHT} * smape",
    "input_size":         INPUT_SIZE,
    "horizon":            HORIZON,
    "n_stacks":           N_STACKS,
    "n_blocks":           N_BLOCKS,
    "hidden_size":        HIDDEN_SIZE,
    "n_layers":           N_LAYERS,
    "pool_kernels":       str(POOL_KERNELS),
    "n_freq_downsample":  str(N_FREQ_DOWNSAMPLE),
    "dropout":            DROPOUT,
    "lr":                 LR,
    "weight_decay":       WEIGHT_DECAY,
    "batch_size":         BATCH_SIZE,
    "max_epochs":         MAX_EPOCHS,
    "grad_clip":          GRAD_CLIP,
    "smape_weight":       SMAPE_WEIGHT,
    "n_train_rows":       len(train),
    "n_val_rows":         len(val),
    "n_test_rows":        len(test),
    "n_train_windows":    len(train_ds),
    "n_val_windows":      len(val_ds),
    "n_stations":         train[GROUP_COL].nunique(),
    "y_mean":             y_mean,
    "y_std":              y_std,
    "train_date_min":     str(train["datetime"].min()),
    "train_date_max":     str(train["datetime"].max()),
    "val_date_min":       str(val["datetime"].min()),
    "val_date_max":       str(val["datetime"].max()),
    "test_date_min":      str(test["datetime"].min()),
    "test_date_max":      str(test["datetime"].max()),
    "n_features":         N_EXOG_PER_STEP,
})
print(f"MLflow run started: {mlflow_run.info.run_id}")

In [ ]:
# =============================================================================
# Training loop
# =============================================================================
from tqdm.notebook import tqdm

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

y_mean_t = torch.tensor(y_mean, device=DEVICE)
y_std_t  = torch.tensor(y_std,  device=DEVICE)

def run_epoch(loader, train_mode: bool):
    model.train(train_mode)
    pct_running   = 0.0
    smape_running = 0.0
    n_batches     = 0
    ctx = torch.enable_grad() if train_mode else torch.no_grad()
    iterator = tqdm(loader, leave=False,
                    desc="train" if train_mode else "val ")
    with ctx:
        for x_lb, x_fu, y_std_b, y_raw_b, prc in iterator:
            x_lb   = x_lb.to(DEVICE, non_blocking=True)
            x_fu   = x_fu.to(DEVICE, non_blocking=True)
            y_std_b= y_std_b.to(DEVICE, non_blocking=True)
            y_raw_b= y_raw_b.to(DEVICE, non_blocking=True)
            prc    = prc.to(DEVICE, non_blocking=True)

            y_pred_std = model(x_lb, x_fu)
            total, m_pct, sm = combined_loss(
                y_pred_std, y_std_b, y_raw_b, prc, y_mean_t, y_std_t
            )

            if train_mode:
                optimizer.zero_grad(set_to_none=True)
                total.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()

            pct_running   += float(m_pct.item())
            smape_running += float(sm.item())
            n_batches     += 1
            iterator.set_postfix(pct=f"{m_pct.item():.3f}",
                                 smape=f"{sm.item():.3f}")

    return pct_running / max(n_batches, 1), smape_running / max(n_batches, 1)


best_val_pct = float("inf")
best_state   = None
history      = []

for epoch in range(1, MAX_EPOCHS + 1):
    train_pct, train_smape = run_epoch(train_loader, train_mode=True)
    val_pct,   val_smape   = run_epoch(val_loader,   train_mode=False)
    scheduler.step()

    print(
        f"epoch {epoch:02d} | "
        f"train_pct={train_pct:.3f}% train_smape={train_smape:.3f} "
        f"val_pct={val_pct:.3f}% val_smape={val_smape:.3f}"
    )

    mlflow.log_metrics({
        "train_money_pct": train_pct,
        "train_smape":     train_smape,
        "val_money_pct":   val_pct,
        "val_smape":       val_smape,
        "lr":              optimizer.param_groups[0]["lr"],
    }, step=epoch)
    history.append((epoch, train_pct, train_smape, val_pct, val_smape))

    if val_pct < best_val_pct:
        best_val_pct = val_pct
        best_state   = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"  ↳ new best val_pct={val_pct:.3f}%")

# Restore best weights for inference
if best_state is not None:
    model.load_state_dict(best_state)
print(f"Best val_money_pct = {best_val_pct:.3f}%")

# Save best checkpoint as MLflow artifact
import os, tempfile
with tempfile.TemporaryDirectory() as tmpd:
    ckpt_path = os.path.join(tmpd, "nhits_best.pt")
    torch.save({
        "state_dict":  best_state if best_state is not None else model.state_dict(),
        "y_mean":      y_mean,
        "y_std":       y_std,
        "input_size":  INPUT_SIZE,
        "horizon":     HORIZON,
    }, ckpt_path)
    mlflow.log_artifact(ckpt_path, artifact_path="checkpoints")
    mlflow.pytorch.log_model(model, artifact_path="model")
print("Best checkpoint logged to MLflow.")

In [ ]:
# =============================================================================
# Recursive inference for a one-month horizon
# =============================================================================
# Each split (val, test) holds ~1 month of hourly data per station, while the
# model only forecasts HORIZON steps at a time. We therefore step through each
# station's series in HORIZON-sized chunks, using the previous chunk's
# PREDICTIONS (not ground truth) as the lookback when stepping forward.
# This avoids future target leakage.
#
# To start, we need INPUT_SIZE hours of seed history. We take it from the END
# of the previous split (train precedes val precedes test in time_idx), so the
# very first prediction has a real lookback rather than zeros.
# =============================================================================

@torch.no_grad()
def recursive_predict(
    seed_df_std: pd.DataFrame,        # standardized seed history per station (>= INPUT_SIZE rows tail)
    target_df_std: pd.DataFrame,      # standardized target window to predict (covariates from here)
    target_df_raw: pd.DataFrame,      # raw version of target window (for prices, datetime, etc.)
) -> pd.DataFrame:
    """
    Returns a DataFrame with columns: GROUP_COL, datetime, time_idx, sum_of_kWh (true raw),
    pred (raw kWh), dam_price, sell_bm_price, buy_bm_price.
    """
    model.eval()
    rows = []

    # Pre-sort everything by station + time
    seed_df_std   = seed_df_std.sort_values([GROUP_COL, "time_idx"]).reset_index(drop=True)
    target_df_std = target_df_std.sort_values([GROUP_COL, "time_idx"]).reset_index(drop=True)
    target_df_raw = target_df_raw.sort_values([GROUP_COL, "time_idx"]).reset_index(drop=True)

    stations = target_df_std[GROUP_COL].unique()
    iterator = tqdm(stations, desc="inference", leave=False)
    for st in iterator:
        seed_y = seed_df_std.loc[seed_df_std[GROUP_COL] == st, Y_COL].to_numpy(dtype=np.float32)
        if len(seed_y) < INPUT_SIZE:
            # Not enough seed history: pad with mean (= 0 in standardized space).
            pad = np.zeros(INPUT_SIZE - len(seed_y), dtype=np.float32)
            seed_y = np.concatenate([pad, seed_y])
        lookback = seed_y[-INPUT_SIZE:].copy()                       # [L]

        tgt_std = target_df_std[target_df_std[GROUP_COL] == st]
        tgt_raw = target_df_raw[target_df_raw[GROUP_COL] == st]
        exog    = tgt_std[EXOG_NUMERIC].to_numpy(dtype=np.float32)   # [N, n_exog]
        n_steps = len(tgt_std)

        preds_raw = np.zeros(n_steps, dtype=np.float32)

        pos = 0
        while pos < n_steps:
            chunk = min(HORIZON, n_steps - pos)
            # Build x_future for this chunk; if chunk < HORIZON, pad with last-row exog
            if chunk < HORIZON:
                pad = np.repeat(exog[-1:], HORIZON - chunk, axis=0)
                x_fu_np = np.concatenate([exog[pos:pos + chunk], pad], axis=0)
            else:
                x_fu_np = exog[pos:pos + HORIZON]

            x_lb = torch.from_numpy(lookback).unsqueeze(0).to(DEVICE)
            x_fu = torch.from_numpy(x_fu_np).unsqueeze(0).to(DEVICE)
            y_pred_std = model(x_lb, x_fu).squeeze(0).cpu().numpy()  # [H]
            y_pred_raw = y_pred_std * y_std + y_mean

            preds_raw[pos:pos + chunk] = y_pred_raw[:chunk]

            # Slide the lookback forward using PREDICTED values (standardized).
            new_std = y_pred_std[:chunk]
            lookback = np.concatenate([lookback[chunk:], new_std])
            pos += chunk

        df_out = pd.DataFrame({
            GROUP_COL:       tgt_raw[GROUP_COL].values,
            "datetime":      tgt_raw["datetime"].values,
            "time_idx":      tgt_raw["time_idx"].values,
            Y_COL:           tgt_raw[Y_COL].values,
            "pred":          preds_raw,
            "dam_price":     tgt_raw["dam_price"].values,
            "sell_bm_price": tgt_raw["sell_bm_price"].values,
            "buy_bm_price":  tgt_raw["buy_bm_price"].values,
        })
        rows.append(df_out)

    return pd.concat(rows, ignore_index=True)


# Seeds: tail of the previous split provides the initial lookback.
val_eval  = recursive_predict(seed_df_std=train_s, target_df_std=val_s,  target_df_raw=val)
test_eval = recursive_predict(seed_df_std=val_s,   target_df_std=test_s, target_df_raw=test)

print(f"val_eval  shape: {val_eval.shape}")
print(f"test_eval shape: {test_eval.shape}")

In [ ]:
def _prices(df):
    return df["dam_price"].values, df["sell_bm_price"].values, df["buy_bm_price"].values

def _mae(y_true, y_pred):
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))

def _bias(y_true, y_pred):
    return float(np.sum(y_pred) - np.sum(y_true))

val_smape_v     = smape(val_eval[Y_COL], val_eval['pred'])
val_rmse_v      = rmse(val_eval[Y_COL], val_eval['pred'])
val_mape_v      = mape(val_eval[Y_COL], val_eval['pred'])
val_money_v     = money(val_eval[Y_COL], val_eval['pred'], *_prices(val_eval))
val_money_pct_v = money_pct(val_eval[Y_COL], val_eval['pred'], *_prices(val_eval))
val_mae_v       = _mae(val_eval[Y_COL], val_eval['pred'])
val_bias_v      = _bias(val_eval[Y_COL], val_eval['pred'])

test_smape_v     = smape(test_eval[Y_COL], test_eval['pred'])
test_rmse_v      = rmse(test_eval[Y_COL], test_eval['pred'])
test_mape_v      = mape(test_eval[Y_COL], test_eval['pred'])
test_money_v     = money(test_eval[Y_COL], test_eval['pred'], *_prices(test_eval))
test_money_pct_v = money_pct(test_eval[Y_COL], test_eval['pred'], *_prices(test_eval))
test_mae_v       = _mae(test_eval[Y_COL], test_eval['pred'])
test_bias_v      = _bias(test_eval[Y_COL], test_eval['pred'])

print("── Validation ──────────────────────────────────────────────")
print(f"Aligned samples : {len(val_eval):,}")
print(f"SMAPE     : {val_smape_v:.4f}")
print(f"MAE       : {val_mae_v:.4f}")
print(f"RMSE      : {val_rmse_v:.4f}")
print(f"MAPE      : {val_mape_v:.2f} %")
print(f"MONEY     : {val_money_v:.4f}")
print(f"MONEY_PCT : {val_money_pct_v:.4f}%")
print(f"BIAS      : {val_bias_v:+.2f} kWh (pred_sum - true_sum)")

print("── Test ────────────────────────────────────────────────────")
print(f"Aligned samples : {len(test_eval):,}")
print(f"SMAPE     : {test_smape_v:.4f}")
print(f"MAE       : {test_mae_v:.4f}")
print(f"RMSE      : {test_rmse_v:.4f}")
print(f"MAPE      : {test_mape_v:.2f} %")
print(f"MONEY     : {test_money_v:.4f}")
print(f"MONEY_PCT : {test_money_pct_v:.4f}%")
print(f"BIAS      : {test_bias_v:+.2f} kWh (pred_sum - true_sum)")

mlflow.log_metrics({
    "val_smape":      val_smape_v,
    "val_mae":        val_mae_v,
    "val_rmse":       val_rmse_v,
    "val_mape":       val_mape_v,
    "val_money":      val_money_v,
    "val_money_pct":  val_money_pct_v,
    "val_bias":       val_bias_v,
    "test_smape":     test_smape_v,
    "test_mae":       test_mae_v,
    "test_rmse":      test_rmse_v,
    "test_mape":      test_mape_v,
    "test_money":     test_money_v,
    "test_money_pct": test_money_pct_v,
    "test_bias":      test_bias_v,
})

# Save sample predictions as artifact
import tempfile, os
with tempfile.TemporaryDirectory() as tmpd:
    sample_csv = os.path.join(tmpd, "test_predictions_sample.csv")
    test_eval.head(50_000).to_csv(sample_csv, index=False)
    mlflow.log_artifact(sample_csv, artifact_path="predictions")

mlflow.end_run()
print(f"MLflow run logged → {mlflow.get_tracking_uri()}")

In [ ]:
def per_station_metrics(eval_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for grp, gdf in eval_df.groupby(GROUP_COL):
        rows.append({
            GROUP_COL:    grp,
            "n":          len(gdf),
            "SMAPE":      smape(gdf[Y_COL], gdf["pred"]),
            "RMSE":       rmse (gdf[Y_COL], gdf["pred"]),
            "MAPE":       mape (gdf[Y_COL], gdf["pred"]),
            "MONEY":      money    (gdf[Y_COL], gdf["pred"], *_prices(gdf)),
            "MONEY_PCT":  money_pct(gdf[Y_COL], gdf["pred"], *_prices(gdf)),
        })
    return pd.DataFrame(rows).sort_values("SMAPE")


test_station_metrics = per_station_metrics(test_eval)

print("Top-10 best stations (test SMAPE):")
print(test_station_metrics.head(10).to_string(index=False))
print("\nBottom-10 worst stations (test SMAPE):")
print(test_station_metrics.tail(10).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


def plot_forecast(df, eic_code, start_dt=None, end_dt=None, title_prefix=""):

    df = df[df[GROUP_COL] == eic_code].sort_values("datetime")
    if df.empty:
        raise ValueError(f"No data for EiC code: {eic_code!r}")
    if start_dt is not None:
        df = df[df["datetime"] >= pd.Timestamp(start_dt)]
    if end_dt is not None:
        df = df[df["datetime"] <= pd.Timestamp(end_dt)]
    if df.empty:
        raise ValueError("No data in the specified datetime range.")

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(df["datetime"], df[Y_COL],  label="True",      linewidth=1, color="steelblue")
    ax.plot(df["datetime"], df["pred"], label="Predicted", linewidth=1, color="tomato", alpha=0.85)
    ax.set_title(
        f"{title_prefix}{eic_code}  |  MAPE={mape(df[Y_COL], df['pred']):.3f}"
        f"  MONEY_PCT={money_pct(df[Y_COL], df['pred'], *_prices(df)):.2f}"
        f"  ({df['datetime'].min().date()} \u2013 {df['datetime'].max().date()})"
    )
    ax.set_xlabel("Datetime")
    ax.set_ylabel(Y_COL)
    ax.legend()
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
    fig.autofmt_xdate(rotation=0, ha="center")
    plt.tight_layout()
    plt.show()


# Example:
# plot_forecast(val_eval,  eic_code="<code>")
# plot_forecast(test_eval, eic_code="<code>", start_dt="2025-08-25", end_dt="2025-08-26")

In [ ]:
best_station  = test_station_metrics.iloc[0][GROUP_COL]
worst_station = test_station_metrics.iloc[-1][GROUP_COL]

print(f"Best  station (SMAPE): {best_station}")
plot_forecast(test_eval, eic_code=best_station,  title_prefix="[BEST]  ")

print(f"Worst station (SMAPE): {worst_station}")
plot_forecast(test_eval, eic_code=worst_station, title_prefix="[WORST] ")

In [ ]:
# =============================================================================
# Inference example: forecast next 48 hours for a single station given its tail
# =============================================================================
# Demonstrates how to run N-HiTS once on a fresh lookback window without the
# recursive loop above.
# =============================================================================

example_station = sampled_stations[0] if "sampled_stations" in dir() else train[GROUP_COL].iloc[0]
ex_df_std = test_s[test_s[GROUP_COL] == example_station].sort_values("time_idx")
ex_df_raw = test  [test  [GROUP_COL] == example_station].sort_values("time_idx")

# seed lookback from end of val for this station
seed = val_s[val_s[GROUP_COL] == example_station].sort_values("time_idx").tail(INPUT_SIZE)
x_lb = torch.from_numpy(seed[Y_COL].to_numpy(dtype=np.float32)).unsqueeze(0).to(DEVICE)
x_fu = torch.from_numpy(ex_df_std[EXOG_NUMERIC].to_numpy(dtype=np.float32)[:HORIZON]).unsqueeze(0).to(DEVICE)

model.eval()
with torch.no_grad():
    y_pred_std = model(x_lb, x_fu).squeeze(0).cpu().numpy()
y_pred_raw = y_pred_std * y_std + y_mean

print(f"Station: {example_station}")
print(f"First 48h prediction (kWh): {y_pred_raw[:5].round(3)} ...")
print(f"Actual first 5h            : {ex_df_raw[Y_COL].to_numpy()[:5].round(3)}")